# 📍 Pose Metric Evaluation (ATE, RTE, RPE)
이 노트북 파일은 `evo` 라이브러리를 활용하여 3DGS 및 LongSplat의 카메라 궤적 예측 결과를 실제 카메라 궤적(GT)과 비교하여 추정 성능(ATE, RTE, RPE)을 평가합니다.
- **ATE (Absolute Trajectory Error)**: 전역 궤적의 절대 위치 오차
- **RTE (Relative Translation Error)**: 상대 병진 위치 오차
- **RPE (Relative Pose Error / RRE)**: 상대 회전 방향 오차 (각도 단위)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install evo pandas tqdm

Mounted at /content/drive


In [6]:
!ls -R /content/drive/MyDrive/3DGS > 3DGS_structure_ls.txt
!ls -R /content/drive/MyDrive/LongSplat > Longsplat_structure_ls.txt
!ls -R /content/drive/MyDrive/GT > GT_structure_ls.txt

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import evo
    from evo.core.trajectory import PoseTrajectory3D
    from evo.tools import file_interface
    from evo.core import sync
    from evo.core import metrics
    from evo.core.metrics import PoseRelation
except ImportError:
    print("Please install evo (e.g. !pip install evo) to compute ATE, RPE.")

In [ ]:
def find_trajectory_file(base_dir):
    """
    구조화된 경로 내에서 카메라 궤적 파일(예: trajectory.txt, cameras.txt 등)을 자동으로 탐색합니다.
    """
    # 1. 파일 이름 패턴 검색 (프로젝트 환경에서 출력되는 궤적 파일명에 맞게 확장자를 수정하세요)
    search_patterns = [
        os.path.join(base_dir, "**", "*traj*.txt"),
        os.path.join(base_dir, "**", "poses*.txt"),
        os.path.join(base_dir, "**", "cameras.txt"),
        os.path.join(base_dir, "**", "images.txt")  # COLMAP 형식의 예시
    ]

    matches = []
    for pattern in search_patterns:
        matches.extend(glob.glob(pattern, recursive=True))

    if matches:
        # 여러 개가 발견될 경우 가장 하위 경로 반환 (혹은 가장 최신)
        return sorted(matches)[-1]

    return None

In [ ]:
def compute_pose_metrics(gt_file, pred_file, align=True):
    """
    GT 궤적과 예측 궤적 간의 ATE, RTE, RPE를 계산합니다.
    (이 기본 함수는 evo 라이브러리와 TUM 궤적 포맷을 기준으로 작성되었습니다.)
    TUM 포맷: [timestamp tx ty tz qx qy qz qw]
    """
    try:
        # 파일 로드 (Kitti 포맷인 경우 read_kitti_poses_file, COLMAP인 경우 evo 내장 변환기 등 적용 필요)
        # COLMAP 텍스트 포맷을 직접 읽으려면 evo_traj colmap 등의 변환 로직 추가 변환 필요할 수 있음
        traj_ref = file_interface.read_tum_trajectory_file(gt_file)
        traj_est = file_interface.read_tum_trajectory_file(pred_file)
    except Exception as e:
        raise ValueError(f"Trajectory 로드 실패. 궤적 파일 포맷에 맞게 수정이 필요할 수 있습니다: {e}")

    # 타임스탬프 기반 궤적 동기화 (만약 타임스탬프가 없다면 인덱스 기반 매칭 필요)
    max_diff = 0.01
    traj_ref, traj_est = sync.associate_trajectories(traj_ref, traj_est, max_diff=max_diff)

    if traj_est.num_poses == 0:
        raise ValueError("GT와 PRED 궤적 간에 동기화되는 포즈가 없습니다.")

    # Scale 및 Pose Alignment 적용 (Umeyama alignment - 3D 모노큘러 카메라에 필수적)
    if align:
        traj_est.align(traj_ref, correct_scale=True, correct_only_scale=False)

    # 1. ATE (Absolute Trajectory Error) - 절대 경로 오차 (위치 차이)
    ape_metric = metrics.APE(PoseRelation.translation_part)
    ape_metric.process_data((traj_ref, traj_est))
    ate_rmse = ape_metric.get_statistic(metrics.StatisticsType.rmse)

    # 2. RTE (Relative Translation Error) - 상대 병진 오차 (1 프레임 이동 기준)
    rpe_trans_metric = metrics.RPE(PoseRelation.translation_part, delta=1, delta_unit=metrics.Unit.frames)
    rpe_trans_metric.process_data((traj_ref, traj_est))
    rte_rmse = rpe_trans_metric.get_statistic(metrics.StatisticsType.rmse)

    # 3. RPE (Relative Pose/Rotation Error) - 상대 회전 오차 (각도 단위)
    rpe_rot_metric = metrics.RPE(PoseRelation.rotation_angle_deg, delta=1, delta_unit=metrics.Unit.frames)
    rpe_rot_metric.process_data((traj_ref, traj_est))
    rre_rmse = rpe_rot_metric.get_statistic(metrics.StatisticsType.rmse)

    return ate_rmse, rte_rmse, rre_rmse

In [ ]:
DIR_BASE = '/content/drive/MyDrive'

METHODS = ["LongSplat", "3DGS"]
CONDITIONS = ["original", "snow", "de_snow"]

# 평가할 Scene 리스트
SCENES = [
    "grass",
    "pillar",
    "road"
]

# GT 디렉토리 및 트래젝토리 탐색
GT_DIR_LIST = [os.path.join(DIR_BASE, "GT", f"gt_{s}") for s in SCENES]
GT_TRAJ_LIST = [find_trajectory_file(gt) for gt in GT_DIR_LIST]

# PRED 디렉토리 및 트래젝토리 탐색
SCENE_TRAJ_LIST = []
for s in SCENES:
    scene_traj = []
    for m in METHODS:
        for c in CONDITIONS:
            base_p = os.path.join(DIR_BASE, m, f"{s}_{c}")
            scene_traj.append(find_trajectory_file(base_p))
    SCENE_TRAJ_LIST.append(scene_traj)

# 파일 탐색 결과 출력
print("=== GT Trajectory Files ===")
for i, gt in enumerate(GT_TRAJ_LIST):
    print(f"{SCENES[i]}: {gt}")

print("\n=== Prediction Trajectory Files ===")
for i, scene_paths in enumerate(SCENE_TRAJ_LIST):
    print(f"\nScene: {SCENES[i]}")
    idx = 0
    for m in METHODS:
        for c in CONDITIONS:
            print(f"  [{m} - {c}]: {scene_paths[idx]}")
            idx += 1

In [ ]:
results = []

for i, (gt_file, pred_group) in enumerate(zip(GT_TRAJ_LIST, SCENE_TRAJ_LIST)):
    scene_name = SCENES[i]
    print(f"\n==============================")
    print(f"GT: {gt_file}\n")

    if not gt_file or not os.path.exists(gt_file):
        print(f"GT 궤적 파일을 찾을 수 없음, 스킵: {gt_file}")
        continue

    path_idx = 0
    for m_name in METHODS:
        for c_name in CONDITIONS:
            pred_file = pred_group[path_idx]
            path_idx += 1

            print(f"PRED ({m_name} - {c_name}): {pred_file}")

            if not pred_file or not os.path.exists(pred_file):
                print(f"PRED 궤적 파일을 찾을 수 없음, 스킵")
                continue

            try:
                # 지표 계산 실행
                ate_val, rte_val, rre_val = compute_pose_metrics(gt_file, pred_file, align=True)

                results.append({
                    'Scene': scene_name,
                    'Method': m_name,
                    'Condition': c_name,
                    'ATE_RMSE': ate_val,
                    'RTE_RMSE': rte_val,
                    'RRE_RMSE_deg': rre_val
                })

                print(f"-> ATE: {ate_val:.4f}, RTE: {rte_val:.4f}, RRE: {rre_val:.4f}°")

            except Exception as e:
                print(f"-> Error: {e}")
                continue

            print()

if results:
    # CSV 저장 및 출력
    df = pd.DataFrame(results)
    csv_path = os.path.join(DIR_BASE, 'pose_metrics_results.csv')
    df.to_csv(csv_path, index=False)
    print(f"\n[DONE] All pose metrics saved to: {csv_path}")
    display(df)
else:
    print("평가된 결과가 없습니다. 궤적 파일(GT 및 PRED) 포맷 또는 경로를 확인하세요.")

In [8]:
import os, glob

OUT_DIR = "/content/drive/MyDrive/pose_eval_results"

for f in glob.glob(os.path.join(OUT_DIR, "*")):
    os.remove(f)

print("pose_eval_results 폴더 파일 삭제 완료")

pose_eval_results 폴더 파일 삭제 완료


In [9]:
# =========================================
# 0) Drive mount + install
# =========================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q evo pandas scipy

# =========================================
# 1) Imports
# =========================================
import os
import json
import glob
import numpy as np
import pandas as pd

from scipy.spatial.transform import Rotation as R
from evo.tools import file_interface
from evo.core import sync, metrics
from evo.core.metrics import PoseRelation

# =========================================
# 2) Config
# =========================================
DIR_BASE = "/content/drive/MyDrive"

METHODS = ["3DGS", "LongSplat"]
SCENES = ["grass", "pillar", "road"]
TARGET_CONDITIONS = ["snow", "de_snow"]

# 이번 평가는 test만
SPLIT = "test"

# 결과 저장 경로
OUT_DIR = os.path.join(DIR_BASE, "pose_eval_results")
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================
# 3) Helper: latest run dir
# =========================================
def latest_run_dir(scene_root: str) -> str:
    """
    예:
    /content/drive/MyDrive/3DGS/grass_original
      └── 2026-04-17_01:34
    이런 식일 때 가장 최신 하위 폴더 반환
    """
    if not os.path.isdir(scene_root):
        raise FileNotFoundError(f"Scene root not found: {scene_root}")

    subdirs = [p for p in glob.glob(os.path.join(scene_root, "*")) if os.path.isdir(p)]
    if not subdirs:
        raise FileNotFoundError(f"No run directory found under: {scene_root}")

    return sorted(subdirs)[-1]

# =========================================
# 4) Helper: pick camera json
# =========================================
def get_camera_json_path(method: str, run_dir: str, split: str = "test") -> str:
    """
    3DGS     -> run_dir/cameras.json
    LongSplat-> run_dir/cameras_all_test.json or cameras_all_train.json
    """
    if method == "3DGS":
        path = os.path.join(run_dir, "cameras.json")
    elif method == "LongSplat":
        path = os.path.join(run_dir, f"cameras_all_{split}.json")
    else:
        raise ValueError(f"Unknown method: {method}")

    if not os.path.exists(path):
        raise FileNotFoundError(f"Camera json not found: {path}")
    return path

# =========================================
# 5) Helper: load camera entries robustly
# =========================================
def load_camera_entries(json_path: str):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # case 1: already list
    if isinstance(data, list):
        return data

    # case 2: dict containing list
    if isinstance(data, dict):
        for key in ["frames", "cameras", "images", "views", "data"]:
            if key in data and isinstance(data[key], list):
                return data[key]

    raise ValueError(f"Unsupported JSON structure: {json_path}")

# =========================================
# 6) Helper: sort key
# =========================================
def sort_key_from_camera(cam: dict, fallback_idx: int):
    """
    img_name, image_name, file_name, name, id 등을 최대한 활용해서 안정적으로 정렬
    """
    for key in ["img_name", "image_name", "file_name", "name"]:
        if key in cam:
            base = os.path.basename(str(cam[key]))
            stem = os.path.splitext(base)[0]
            digits = "".join(ch for ch in stem if ch.isdigit())
            if digits:
                return (0, int(digits))
            return (1, stem)

    if "id" in cam:
        try:
            return (2, int(cam["id"]))
        except:
            return (3, str(cam["id"]))

    return (4, fallback_idx)

# =========================================
# 7) Helper: camera json -> pose (t, q)
# =========================================
def camera_to_t_q(cam):
    """
    지원 형식:
    1) 3DGS style:
       {
         "position": [...],
         "rotation": [[...],[...],[...]]
       }

    2) LongSplat style:
       {
         "uid": ...,
         "R": [[...],[...],[...]],
         "T": [...],
         ...
       }
    """

    # ---------------------------------
    # A) 3DGS style: position + rotation
    # ---------------------------------
    if "position" in cam and "rotation" in cam:
        t = np.asarray(cam["position"], dtype=float).reshape(3)
        rot = np.asarray(cam["rotation"], dtype=float).reshape(3, 3)
        q = R.from_matrix(rot).as_quat()   # [qx, qy, qz, qw]
        return t, q

    # ---------------------------------
    # B) LongSplat style: R + T
    # graphdeco 계열 카메라 포맷과 맞춰서 변환
    # ---------------------------------
    if "R" in cam and "T" in cam:
        Rt = np.eye(4, dtype=float)

        # camera.R / camera.T 형식을 W2C로 복원
        Rt[:3, :3] = np.asarray(cam["R"], dtype=float).reshape(3, 3).T
        Rt[:3, 3] = np.asarray(cam["T"], dtype=float).reshape(3)

        # C2W로 변환
        C2W = np.linalg.inv(Rt)

        t = C2W[:3, 3]
        rot = C2W[:3, :3]
        q = R.from_matrix(rot).as_quat()
        return t, q

    # ---------------------------------
    # C) 4x4 matrix style
    # ---------------------------------
    for key in ["transform_matrix", "c2w", "camera_to_world"]:
        if key in cam:
            mat = np.asarray(cam[key], dtype=float).reshape(4, 4)
            t = mat[:3, 3]
            rot = mat[:3, :3]
            q = R.from_matrix(rot).as_quat()
            return t, q

    raise ValueError(f"Unsupported camera pose fields. keys={list(cam.keys())}")

# =========================================
# 8) Helper: export json -> TUM trajectory
# =========================================
def export_json_to_tum(json_path: str, tum_out_path: str):
    cams = load_camera_entries(json_path)

    parsed = []
    for i, cam in enumerate(cams):
        t, q = camera_to_t_q(cam)
        parsed.append((sort_key_from_camera(cam, i), t, q))

    # stable sort
    parsed = sorted(parsed, key=lambda x: x[0])

    with open(tum_out_path, "w", encoding="utf-8") as f:
        for idx, (_, t, q) in enumerate(parsed):
            ts = float(idx)
            f.write(
                f"{ts:.6f} "
                f"{t[0]:.10f} {t[1]:.10f} {t[2]:.10f} "
                f"{q[0]:.10f} {q[1]:.10f} {q[2]:.10f} {q[3]:.10f}\n"
            )

# =========================================
# 9) Pose metric
# =========================================
def compute_pose_metrics(gt_tum_file: str, pred_tum_file: str, align: bool = True):
    traj_ref = file_interface.read_tum_trajectory_file(gt_tum_file)
    traj_est = file_interface.read_tum_trajectory_file(pred_tum_file)

    # timestamp association
    traj_ref, traj_est = sync.associate_trajectories(traj_ref, traj_est, max_diff=0.01)

    if traj_ref.num_poses == 0 or traj_est.num_poses == 0:
        raise ValueError("No synchronized poses found between GT and prediction.")

    # alignment
    if align:
        traj_est.align(traj_ref, correct_scale=True, correct_only_scale=False)

    # ATE
    ape_metric = metrics.APE(PoseRelation.translation_part)
    ape_metric.process_data((traj_ref, traj_est))
    ate_rmse = ape_metric.get_statistic(metrics.StatisticsType.rmse)

    # Relative translation error
    rpe_trans_metric = metrics.RPE(
        PoseRelation.translation_part,
        delta=1,
        delta_unit=metrics.Unit.frames
    )
    rpe_trans_metric.process_data((traj_ref, traj_est))
    rte_rmse = rpe_trans_metric.get_statistic(metrics.StatisticsType.rmse)

    # Relative rotation error (deg)
    rpe_rot_metric = metrics.RPE(
        PoseRelation.rotation_angle_deg,
        delta=1,
        delta_unit=metrics.Unit.frames
    )
    rpe_rot_metric.process_data((traj_ref, traj_est))
    rre_rmse = rpe_rot_metric.get_statistic(metrics.StatisticsType.rmse)

    return ate_rmse, rte_rmse, rre_rmse

# =========================================
# 10) Main evaluation
# =========================================
results = []

for method in METHODS:
    print(f"\n==================== {method} ====================")

    for scene in SCENES:
        try:
            gt_scene_root = os.path.join(DIR_BASE, method, f"{scene}_original")
            gt_run_dir = latest_run_dir(gt_scene_root)
            gt_json = get_camera_json_path(method, gt_run_dir, split=SPLIT)

            gt_tum = os.path.join(OUT_DIR, f"{method}_{scene}_original_{SPLIT}.tum")
            export_json_to_tum(gt_json, gt_tum)

            print(f"\n[GT] {method} | {scene} | original")
            print(f"run_dir: {gt_run_dir}")
            print(f"json   : {gt_json}")

        except Exception as e:
            print(f"\n[GT ERROR] {method} | {scene} -> {e}")
            continue

        for cond in TARGET_CONDITIONS:
            try:
                pred_scene_root = os.path.join(DIR_BASE, method, f"{scene}_{cond}")
                pred_run_dir = latest_run_dir(pred_scene_root)
                pred_json = get_camera_json_path(method, pred_run_dir, split=SPLIT)

                pred_tum = os.path.join(OUT_DIR, f"{method}_{scene}_{cond}_{SPLIT}.tum")
                export_json_to_tum(pred_json, pred_tum)

                ate, rte, rre = compute_pose_metrics(gt_tum, pred_tum, align=True)

                results.append({
                    "Method": method,
                    "Scene": scene,
                    "GT_Condition": "original",
                    "Pred_Condition": cond,
                    "Split": SPLIT,
                    "ATE_RMSE": ate,
                    "RTE_RMSE": rte,
                    "RRE_RMSE_deg": rre,
                    "GT_RunDir": gt_run_dir,
                    "Pred_RunDir": pred_run_dir,
                    "GT_Json": gt_json,
                    "Pred_Json": pred_json
                })

                print(f"[OK] {scene} | {cond} -> "
                      f"ATE={ate:.6f}, RTE={rte:.6f}, RRE={rre:.6f}")

            except Exception as e:
                print(f"[ERROR] {method} | {scene} | {cond} -> {e}")

# =========================================
# 11) Save result
# =========================================
if len(results) == 0:
    print("\nNo results were computed.")
else:
    df = pd.DataFrame(results)
    display(df)

    csv_path = os.path.join(OUT_DIR, f"pose_metrics_results_{SPLIT}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"\n[DONE] saved to: {csv_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

==================== 3DGS ====================

[GT] 3DGS | grass | original
run_dir: /content/drive/MyDrive/3DGS/grass_original/2026-04-17_01:34
json   : /content/drive/MyDrive/3DGS/grass_original/2026-04-17_01:34/cameras.json
[OK] grass | snow -> ATE=0.000592, RTE=0.000245, RRE=0.006403
[OK] grass | de_snow -> ATE=0.000504, RTE=0.000292, RRE=0.007766

[GT] 3DGS | pillar | original
run_dir: /content/drive/MyDrive/3DGS/pillar_original/2026-04-17_14:44
json   : /content/drive/MyDrive/3DGS/pillar_original/2026-04-17_14:44/cameras.json
[OK] pillar | snow -> ATE=0.002537, RTE=0.000475, RRE=0.008777
[OK] pillar | de_snow -> ATE=0.002121, RTE=0.000371, RRE=0.006215

[GT] 3DGS | road | original
run_dir: /content/drive/MyDrive/3DGS/road_original/2026-04-18_04:14
json   : /content/drive/MyDrive/3DGS/road_original/2026-04-18_04:14/cameras.json
[OK] road | snow -> ATE=

,Method,Scene,GT_Condition,Pred_Condition,Split,ATE_RMSE,RTE_RMSE,RRE_RMSE_deg,GT_RunDir,Pred_RunDir,GT_Json,Pred_Json
0,3DGS,grass,original,snow,test,0.000592,0.000245,0.006403,/content/drive/MyDrive/3DGS/grass_original/202...,/content/drive/MyDrive/3DGS/grass_snow/2026-04...,/content/drive/MyDrive/3DGS/grass_original/202...,/content/drive/MyDrive/3DGS/grass_snow/2026-04...
1,3DGS,grass,original,de_snow,test,0.000504,0.000292,0.007766,/content/drive/MyDrive/3DGS/grass_original/202...,/content/drive/MyDrive/3DGS/grass_de_snow/2026...,/content/drive/MyDrive/3DGS/grass_original/202...,/content/drive/MyDrive/3DGS/grass_de_snow/2026...
2,3DGS,pillar,original,snow,test,0.002537,0.000475,0.008777,/content/drive/MyDrive/3DGS/pillar_original/20...,/content/drive/MyDrive/3DGS/pillar_snow/2026-0...,/content/drive/MyDrive/3DGS/pillar_original/20...,/content/drive/MyDrive/3DGS/pillar_snow/2026-0...
3,3DGS,pillar,original,de_snow,test,0.002121,0.000371,0.006215,/content/drive/MyDrive/3DGS/pillar_original/20...,/content/drive/MyDrive/3DGS/pillar_de_snow/202...,/content/drive/MyDrive/3DGS/pillar_original/20...,/content/drive/MyDrive/3DGS/pillar_de_snow/202...
4,3DGS,road,original,snow,test,0.006439,0.005025,0.022020,/content/drive/MyDrive/3DGS/road_original/2026...,/content/drive/MyDrive/3DGS/road_snow/2026-04-...,/content/drive/MyDrive/3DGS/road_original/2026...,/content/drive/MyDrive/3DGS/road_snow/2026-04-...
5,3DGS,road,original,de_snow,test,0.003356,0.003242,0.011352,/content/drive/MyDrive/3DGS/road_original/2026...,/content/drive/MyDrive/3DGS/road_de_snow/2026-...,/content/drive/MyDrive/3DGS/road_original/2026...,/content/drive/MyDrive/3DGS/road_de_snow/2026-...
6,LongSplat,grass,original,snow,test,0.525718,0.670398,9.418189,/content/drive/MyDrive/LongSplat/grass_origina...,/content/drive/MyDrive/LongSplat/grass_snow/20...,/content/drive/MyDrive/LongSplat/grass_origina...,/content/drive/MyDrive/LongSplat/grass_snow/20...
7,LongSplat,grass,original,de_snow,test,0.365003,0.483744,6.078653,/content/drive/MyDrive/LongSplat/grass_origina...,/content/drive/MyDrive/LongSplat/grass_de_snow...,/content/drive/MyDrive/LongSplat/grass_origina...,/content/drive/MyDrive/LongSplat/grass_de_snow...
8,LongSplat,pillar,original,snow,test,0.035621,0.034801,1.007487,/content/drive/MyDrive/LongSplat/pillar_origin...,/content/drive/MyDrive/LongSplat/pillar_snow/2...,/content/drive/MyDrive/LongSplat/pillar_origin...,/content/drive/MyDrive/LongSplat/pillar_snow/2...
9,LongSplat,pillar,original,de_snow,test,0.030187,0.021369,0.410690,/content/drive/MyDrive/LongSplat/pillar_origin...,/content/drive/MyDrive/LongSplat/pillar_de_sno...,/content/drive/MyDrive/LongSplat/pillar_origin...,/content/drive/MyDrive/LongSplat/pillar_de_sno...



[DONE] saved to: /content/drive/MyDrive/pose_eval_results/pose_metrics_results_test.csv
